# Model V0a — AR(1) Only (Baseline)

**Likelihood:**
$$y_{i,1} \sim \mathcal{N}(\alpha_i,\, \tau_i)$$
$$y_{i,t} \sim \mathcal{N}\Big(\alpha_i + \phi\,(y_{i,t-1} - \alpha_i),\, \tau_i\Big), \quad t = 2,\ldots,T$$

**Mean function:** $\mu_{i,t} = \alpha_i$ (constant intercept, no seasonality, no events)

**Priors:**
- $\alpha_i \sim \mathcal{N}(0, 0.001)$
- $\tau_i \sim \Gamma(0.001, 0.001)$
- $\phi \sim \mathrm{Uniform}(-1, 1)$

In [ ]:
from model_helpers import *

VERSION = 'v0a'
PLOT_DIR = f'../../data/models/{VERSION}/plots/'
DATA_DIR = f'../../data/models/{VERSION}/'

import os
os.makedirs(PLOT_DIR, exist_ok=True)

## Load the model parameters

In [ ]:
df_og, raw_df, regions, n_region = load_model_data(VERSION)
n_weeks = df_og.shape[1] - 1
df_mu, df_mu_lower, df_mu_upper = reconstruct_mu_baseline(raw_df, regions, n_weeks)
phi_mean = raw_df['phi'].mean()

## Preprocess
Transpose the og df into regions (cols) x time (weeks)

In [ ]:
df_og = transpose_observed(df_og)

## Make the model estimate df
Formula: `y[i,t] ~ dnorm(alpha[i] + phi * (y[i,t-1] - alpha[i]), tau[i])`

In [ ]:
df_ar1 = compute_ar1_fitted(df_mu, df_og, phi_mean)
df_ar1

## Plot MU
For V0a this is a flat line per region (alpha_i only — no seasonality)

In [ ]:
plot_mu(df_mu, df_mu_lower, df_mu_upper, PLOT_DIR + 'mu_fit.png')

## Plot model fit

In [ ]:
plot_ar1_fit(df_ar1, PLOT_DIR + 'ar1_fit.png')

## Plot the residuals

In [ ]:
df_residuals, df_std_resid = compute_residuals(df_og, df_ar1)

In [ ]:
plot_residuals_ts(df_std_resid, PLOT_DIR + 'residuals_ts.png')

### Autocorrelation of Residuals

In [ ]:
plot_residuals_acf(df_std_resid, PLOT_DIR + 'residuals_acf.png')

### Residuals vs Fitted

In [ ]:
plot_residuals_vs_fitted(df_std_resid, df_ar1, PLOT_DIR + 'residuals_vs_fitted.png')

### QQ-Plot For Residuals

In [ ]:
plot_residuals_qq(df_std_resid, PLOT_DIR + 'residuals_qq.png')

### Residual Periodogram

In [ ]:
plot_residuals_periodogram(df_std_resid, PLOT_DIR + 'residuals_periodogram.png')

# Significance Testing

## Global parameter summaries

In [ ]:
global_params = {
    'phi': raw_df['phi'].values,
}
summarize_global_parameters(raw_df, list(global_params.keys()))

## Alpha baseline
Region intercepts — the only structural parameter in V0a.

In [ ]:
results = []
for i, region in enumerate(regions):
    vals = raw_df[f'alpha[{i+1}]'].values
    ci_lower, ci_upper = np.quantile(vals, [0.025, 0.975])
    results.append({
        'Region': region,
        'Mean': vals.mean(),
        'SD': vals.std(),
        '2.5%': ci_lower,
        '97.5%': ci_upper,
    })
pd.DataFrame(results).round(4)

### Alpha baseline — pairwise test
Tests on $\alpha_i$ directly (the region intercepts).

In [ ]:
alpha_samples = {regions[i]: raw_df[f'alpha[{i+1}]'].values for i in range(n_region)}
df_alpha_pw = pairwise_test(alpha_samples, regions)
df_alpha_pw.to_csv(DATA_DIR + 'alpha_pairwise.csv', index=False)
df_alpha_pw

In [ ]:
pairwise_heatmap(df_alpha_pw, 'P(diff > 0)',
                 r'Alpha ($\alpha_i$): P(Region A > Region B)',
                 PLOT_DIR + 'alpha_pairwise.png')

# Ranking
Ranked on $\alpha_i$ (region intercept) per MCMC draw.

In [ ]:
ranked_alpha = build_ranked_alpha(raw_df, regions)

### Tables

In [ ]:
mean_ranks = ranked_alpha.mean(axis=0)
assigned_rank = mean_ranks.rank().astype(int)

rows = []
for region in ranked_alpha.columns:
    counts = ranked_alpha[region].value_counts().reindex(range(1, 7), fill_value=0)
    row = {'Region': SHORT_NAMES.get(region, region),
           'Mean Rank': mean_ranks[region],
           'Assigned': assigned_rank[region]}
    for r in range(1, 7):
        row[f'P(rank={r})'] = counts[r] / len(ranked_alpha)
    rows.append(row)
pd.DataFrame(rows).round(3)

### Distributions of ranks

In [ ]:
n_samples = len(ranked_alpha)
fig, axes = plt.subplots(2, 3, figsize=(12, 6), dpi=150, layout='constrained', sharey=True)

for ax, col in zip(axes.flatten(), ranked_alpha.columns):
    counts = ranked_alpha[col].value_counts().reindex(range(1, 7), fill_value=0)
    ax.bar(counts.index, counts.values / n_samples)
    ax.set_title(SHORT_NAMES.get(col, col), fontsize=10)
    ax.set_xlabel('Rank')
    ax.set_ylabel('P(rank)')

fig.suptitle('Rank Distributions (V0a — AR(1) Only)')
fig.savefig(PLOT_DIR + 'rank_distributions.png', bbox_inches='tight', dpi=150)
plt.show()